# NorthRiver Warehouse – Data Dictionary

Five core tables in the `northriver` warehouse (Databricks / Unity Catalog). Slowly changing dimensions are implemented as SCD Type 2 (full history preserved via surrogate keys and validity windows). Pipeline notebook references follow the standard NorthRiver naming convention — update the paths if your notebooks differ.

---

## fact_orders

**Grain:** one row per order line item (a single product on a single order).

| Column | Data Type | Description |
|---|---|---|
| order_line_id | BIGINT | Primary key; unique identifier for the order line. |
| order_id | BIGINT | Natural key of the parent order; one order may have many lines. |
| order_line_number | INT | Position of this line within the order (1-based). |
| customer_key | BIGINT | Foreign key to `dim_customer.customer_key` (SCD2 surrogate). |
| product_key | BIGINT | Foreign key to `dim_product.product_key` (SCD2 surrogate). |
| channel_key | INT | Foreign key to `dim_channel.channel_key`. |
| order_date_key | INT | Foreign key to `dim_date.date_key` for the order date. |
| ship_date_key | INT | Foreign key to `dim_date.date_key` for the ship date (NULL if not shipped). |
| quantity_ordered | INT | Number of units ordered on this line. |
| quantity_shipped | INT | Number of units actually shipped (NULL if not shipped). |
| unit_price | DECIMAL(12,2) | Price per unit at time of sale. |
| discount_amount | DECIMAL(12,2) | Discount applied to the line (absolute currency amount). |
| extended_amount | DECIMAL(12,2) | Gross line amount: quantity_ordered × unit_price. |
| net_amount | DECIMAL(12,2) | Extended amount less discount; the revenue recognized for the line. |
| order_status | STRING | Line status, e.g. `created`, `shipped`, `delivered`, `returned`, `cancelled`. |
| _rescued_data | STRING | Capture of unrecognized/malformed source payload for the line, if any. |
| _ingested_at | TIMESTAMP | Timestamp when the row was loaded into the warehouse. |

**Populated by:** `nb_load_fact_orders` (incremental merge from curated order-line events). 
*Note: `_rescued_data` and `_ingested_at` are optional/PII-adjacent technical columns — include only if your fact table carries them; drop the rows from the table above otherwise.*

---

## dim_customer (SCD Type 2)

**Grain:** one row per customer per version — a new row is created each time a tracked customer attribute changes.

| Column | Data Type | Description |
|---|---|---|
| customer_key | BIGINT | Surrogate primary key; unique per customer version. |
| customer_id | BIGINT | Natural/business key of the customer from the source system. |
| customer_name | STRING | Full customer name. |
| email | STRING | Primary contact email address. |
| customer_segment | STRING | Commercial segment, e.g. `retail`, `wholesale`, `online`. |
| customer_region | STRING | Geographic region, e.g. `north`, `south`, `east`, `west`. |
| is_current | BOOLEAN | TRUE for the active version of the customer; exactly one per `customer_id`. |
| effective_from | TIMESTAMP | Start of the version's validity window. |
| effective_to | TIMESTAMP | End of the validity window; NULL for the current version. |

**Populated by:** `nb_load_dim_customer` (SCD2 merge — closes out the prior version, inserts the new one).

---

## dim_product (SCD Type 2)

**Grain:** one row per product per version — a new row is created each time a tracked product attribute changes.

| Column | Data Type | Description |
|---|---|---|
| product_key | BIGINT | Surrogate primary key; unique per product version. |
| product_id | BIGINT | Natural/business key of the product from the source system. |
| product_name | STRING | Display name of the product. |
| category | STRING | Top-level product category, e.g. `apparel`, `footwear`, `equipment`. |
| subcategory | STRING | Product subcategory within the category. |
| brand | STRING | Brand of the product. |
| unit_cost | DECIMAL(12,2) | Cost per unit; tracked as an SCD2 attribute. |
| list_price | DECIMAL(12,2) | List price per unit; tracked as an SCD2 attribute. |
| is_current | BOOLEAN | TRUE for the active version of the product; exactly one per `product_id`. |
| effective_from | TIMESTAMP | Start of the version's validity window. |
| effective_to | TIMESTAMP | End of the validity window; NULL for the current version. |

**Populated by:** `nb_load_dim_product` (SCD2 merge — closes out the prior version, inserts the new one).

---

## dim_channel

**Grain:** one row per sales channel; a stable lookup dimension with no SCD (attributes are overwritten in place).

| Column | Data Type | Description |
|---|---|---|
| channel_key | INT | Primary key; stable surrogate key of the channel. |
| channel_code | STRING | Short natural/business code, e.g. `web`, `retail`, `partner`. |
| channel_name | STRING | Human-readable channel name. |
| channel_description | STRING | Longer description of the channel's purpose. |
| is_active | BOOLEAN | TRUE if the channel is currently in use. |

**Populated by:** `nb_load_dim_channel` (full refresh merge on `channel_key`).

---

## dim_date

**Grain:** one row per calendar date; a static, pre-generated dimension with no SCD.

| Column | Data Type | Description |
|---|---|---|
| date_key | INT | Primary key; integer in `yyyymmdd` format (e.g. 20250131). |
| full_date | DATE | The actual calendar date. |
| day | INT | Day of month (1–31). |
| month | INT | Calendar month number (1–12). |
| month_name | STRING | Name of the month, e.g. `January`. |
| quarter | INT | Calendar quarter (1–4). |
| year | INT | Calendar year (e.g. 2025). |
| day_of_week | INT | Day of week (1 = Sunday … 7 = Saturday). |
| day_name | STRING | Name of the day, e.g. `Monday`. |
| is_weekend | BOOLEAN | TRUE for Saturday and Sunday. |
| is_holiday | BOOLEAN | TRUE for recognized holidays (US calendar). |

**Populated by:** `nb_load_dim_date` (one-time generation of the calendar range; re-run only to extend it).


# Pipeline Lineage: PostgreSQL OLTP → Staging → Warehouse

Data moves from the NorthRiver PostgreSQL OLTP database through a staging layer into the conformed warehouse in the `northriver` schema. Every stage is owned by one notebook in the NorthRiver pipeline; each arrow below names that notebook and the main transformation it applies.

```text
PostgreSQL OLTP (northriver)
    │
    ├── customers ──────────────── nb_extract_customers (extract: full pull w/ change detection)
    │                                   └──> staging.stg_customers
    ├── products ────────────────── nb_extract_products (extract: full pull w/ change detection)
    │                                   └──> staging.stg_products
    ├── channels ────────────────── nb_extract_channels (extract: full snapshot)
    │                                   └──> staging.stg_channels
    ├── orders ───────────────────── nb_extract_orders (extract: incremental pull on updated_at)
    │                                   └──> staging.stg_orders
    └── order_lines ─────────────── nb_extract_order_lines (extract: incremental pull, capture malformed payload in _rescued_data)
                                        └──> staging.stg_order_lines

staging (Bronze/Silver landing tables, append-only)
    │
    ├── stg_customers ──→ nb_load_dim_customer (SCD Type 2 merge: close prior version, insert new) ──→ northriver.dim_customer
    ├── stg_products ───→ nb_load_dim_product   (SCD Type 2 merge: close prior version, insert new) ──→ northriver.dim_product
    ├── stg_channels ───→ nb_load_dim_channel   (full refresh merge on channel_key)                ──→ northriver.dim_channel
    │                        (dim_date is not extracted from OLTP: nb_load_dim_date generates the calendar range one-time)
    └── stg_orders + stg_order_lines
                         └──> nb_load_fact_orders (point-in-time join: order/ship timestamps
                              resolved against the SCD2 validity windows of dim_customer and
                              dim_product to pick the correct surrogate keys; incremental merge
                              into the fact grain of one row per order line)
                                └──> northriver.fact_orders
```

## Step-by-step summary

| # | Step | Source | Target | Notebook | Main transformation |
|---|---|---|---|---|---|
| 1 | Extract customers | OLTP `customers` | `staging.stg_customers` | `nb_extract_customers` | Extract (full pull with change detection) |
| 2 | Extract products | OLTP `products` | `staging.stg_products` | `nb_extract_products` | Extract (full pull with change detection) |
| 3 | Extract channels | OLTP `channels` | `staging.stg_channels` | `nb_extract_channels` | Extract (full snapshot) |
| 4 | Extract orders | OLTP `orders` | `staging.stg_orders` | `nb_extract_orders` | Extract (incremental on `updated_at`) |
| 5 | Extract order lines | OLTP `order_lines` | `staging.stg_order_lines` | `nb_extract_order_lines` | Extract (incremental; malformed payloads captured in `_rescued_data`) |
| 6 | Load dim_customer | `stg_customers` | `dim_customer` | `nb_load_dim_customer` | SCD Type 2 merge (close prior version, insert new) |
| 7 | Load dim_product | `stg_products` | `dim_product` | `nb_load_dim_product` | SCD Type 2 merge (close prior version, insert new) |
| 8 | Load dim_channel | `stg_channels` | `dim_channel` | `nb_load_dim_channel` | Full refresh merge on `channel_key` |
| 9 | Load dim_date | — | `dim_date` | `nb_load_dim_date` | One-time calendar generation (re-run only to extend range) |
| 10 | Load fact_orders | `stg_orders` + `stg_order_lines` + dims | `fact_orders` | `nb_load_fact_orders` | Point-in-time join to SCD2 dims (select correct `customer_key` / `product_key` by validity window), then incremental merge |

## Notes

- **Dimension dependencies:** `nb_load_fact_orders` must run *after* `nb_load_dim_customer`, `nb_load_dim_product`, `nb_load_dim_channel`, and `nb_load_dim_date` so the point-in-time join resolves against current dimension versions.
- **Surrogate key correctness:** because `dim_customer` and `dim_product` are SCD Type 2, the fact join must match on the natural key **and** the order/ship timestamp falling within `effective_from` / `effective_to` — this is what guarantees historically accurate keys in `fact_orders`.
- **Naming:** notebook paths follow the standard NorthRiver convention; update the names above if your pipeline notebooks differ.